# MSigDB decoupleR GSEA Analysis: pathway coverage across sample sizes

**Environment:** `clamp-analyses`

For each CLAMP model (CLAMPfull and CLAMPbase) across coverage levels, this notebook:

1. Loads the Z matrix (gene loadings per LV) for a given coverage level/seed.
2. Filters genes to the model gene universe overlapping MSigDB (v2026.1).
3. Runs `decoupleR::run_fgsea()` on the full loading matrix (all LVs as columns), using the raw LV loadings as the ranking statistic (no top-N filtering, no absolute value).
4. Keeps only **positive-side** enrichment (`statistic == "norm_fgsea"` and `score > 0`) — i.e. MSigDB terms enriched among genes with high *positive* LV loadings. The negative tail is discarded entirely.
5. Stores raw `terms_padj`: the minimum BH-adjusted p-value per MSigDB term across all LVs (BH-adjustment done within each LV, then minimum taken across LVs — same convention as the decoupleR ORA sibling notebook).
6. Saves per-seed RDS caches (`rs{pct}_seed{seed}_msigdb_decoupler_gsea.rds`) and per-pct-level summary RDS/CSV. FDR thresholds and coverage computation are done in `01_msigdb_decoupler_gsea_plot.ipynb`.

At 100% coverage there is no subsampling (seed dirs are identical), so only seed 1 is run for that level to avoid redundant computation.

In [ ]:
library(here)
library(dplyr)
library(decoupleR)

## Paths

In [ ]:
models_dir <- here("output/01_model_building/04_archs4/06_bp_coverage_rshall")
output_dir <- here("output/03_model_biology/00_archs4/05_coverage_random/decoupler_gsea")

dir.create(file.path(output_dir, "CLAMPfull"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(output_dir, "CLAMPbase"),  recursive = TRUE, showWarnings = FALSE)

## Coverage level specs

Full grid: 1/5/10/25/50/75/100%, seeds 1-3. At 100% there is no subsampling (all 3 seed dirs contain identical data), so only seed 1 is run for that level.

In [ ]:
coverage_specs <- list(
  list(pct = 1,   dir = "00_bp_coverage_hall_rs_01"),
  list(pct = 5,   dir = "01_bp_coverage_hall_rs_05"),
  list(pct = 10,  dir = "02_bp_coverage_hall_rs_10"),
  list(pct = 25,  dir = "03_bp_coverage_hall_rs_25"),
  list(pct = 50,  dir = "04_bp_coverage_hall_rs_50"),
  list(pct = 75,  dir = "05_bp_coverage_hall_rs_75"),
  list(pct = 100, dir = "06_bp_coverage_hall_rs_100")
)

seeds <- 1:3

## Load MSigDB gene sets as a decoupleR network

In [ ]:
msig_gmt <- clusterProfiler::read.gmt(here("data/pathways/msigdb.v2026.1.Hs.symbols.gmt"))
net <- msig_gmt %>% dplyr::rename(source = term, target = gene)
message(sprintf("MSigDB gene sets loaded: %d", length(unique(net$source))))

## Helper: run decoupleR GSEA for one model

Returns a list with raw `terms_padj` (minimum BH-adjusted p-value per MSigDB term across all LVs), positive-side only.

In [ ]:
run_gsea_for_model <- function(z_path, n_cores = 4) {
  Z              <- read.csv(z_path, row.names = 1, check.names = FALSE)
  universe_genes <- rownames(Z)
  mat            <- as.matrix(Z[universe_genes %in% net$target, , drop = FALSE])
  n_lvs          <- ncol(mat)

  term_overlap   <- tapply(net$target %in% rownames(mat), net$source, sum)
  n_total_msigdb <- sum(term_overlap >= 10L)

  gsea_res <- decoupleR::run_fgsea(mat = mat, network = net, minsize = 10, nproc = n_cores)

  # positive-side only: enrichment among genes with high positive LV loadings
  gsea_res <- gsea_res %>% dplyr::filter(statistic == "norm_fgsea", score > 0)

  # BH-adjust within each LV (condition), then take min adjusted p per pathway across LVs
  gsea_res <- gsea_res %>%
    dplyr::group_by(condition) %>%
    dplyr::mutate(p_adj = p.adjust(p_value, method = "BH")) %>%
    dplyr::ungroup()

  # n_samples via B.csv header only (avoids loading the full multi-GB model rds)
  b_path <- file.path(dirname(z_path), "B.csv")
  n_samples <- if (file.exists(b_path)) {
    ncol(read.csv(b_path, nrows = 0, check.names = FALSE))
  } else NA_integer_

  list(
    n_samples      = n_samples,
    n_lvs          = n_lvs,
    n_total_msigdb = n_total_msigdb,
    terms_padj     = tapply(gsea_res$p_adj, gsea_res$source, min)
  )
}

## Run GSEA: CLAMPfull

In [ ]:
results_clampfull_by_pct <- list()

for (spec in coverage_specs) {
  pct_rds_path <- file.path(output_dir, "CLAMPfull", sprintf("results_pct%d_msigdb_decoupler_gsea.rds", spec$pct))

  if (file.exists(pct_rds_path)) {
    message(sprintf("Loading cached pct-level result: CLAMPfull %d%%", spec$pct))
    results_clampfull_by_pct[[as.character(spec$pct)]] <- readRDS(pct_rds_path)
    next
  }

  seeds_to_run <- if (spec$pct == 100) 1 else seeds

  pct_rows <- lapply(seeds_to_run, function(s) {
    seed_dir   <- file.path(models_dir, spec$dir, sprintf("hall_coverage_rs%d_seed_%d", spec$pct, s))
    z_path     <- file.path(seed_dir, "CLAMPfull_hall", "Z.csv")
    cache_path <- file.path(output_dir, "CLAMPfull", sprintf("rs%d_seed%d_msigdb_decoupler_gsea.rds", spec$pct, s))

    if (!file.exists(z_path)) { warning("Z.csv not found: ", z_path); return(NULL) }

    if (file.exists(cache_path)) {
      message(sprintf("Loading cached: CLAMPfull rs%d seed%d", spec$pct, s))
      res <- readRDS(cache_path)
    } else {
      message(sprintf("Running decoupleR GSEA: CLAMPfull rs%d seed%d", spec$pct, s))
      res <- run_gsea_for_model(z_path)
      if (!is.null(res)) saveRDS(res, cache_path)
    }
    if (is.null(res)) return(NULL)

    data.frame(
      model_type     = "CLAMPfull",
      coverage_pct   = spec$pct,
      seed           = s,
      n_samples      = res$n_samples,
      n_lvs          = res$n_lvs,
      n_total_msigdb = res$n_total_msigdb,
      stringsAsFactors = FALSE
    )
  })

  pct_df <- do.call(rbind, Filter(Negate(is.null), pct_rows))
  rownames(pct_df) <- NULL
  results_clampfull_by_pct[[as.character(spec$pct)]] <- pct_df
  saveRDS(pct_df, pct_rds_path)
  message(sprintf("Saved: CLAMPfull %d%% -> %s", spec$pct, pct_rds_path))
}

results_clampfull_df <- do.call(rbind, results_clampfull_by_pct)
rownames(results_clampfull_df) <- NULL
print(results_clampfull_df)

In [ ]:
for (pct in names(results_clampfull_by_pct)) {
  csv_path <- file.path(output_dir, "CLAMPfull", sprintf("results_pct%s_msigdb_decoupler_gsea.csv", pct))
  write.csv(results_clampfull_by_pct[[pct]], csv_path, row.names = FALSE)
  message("Saved: ", csv_path)
}

## Run GSEA: CLAMPbase

In [ ]:
results_clampbase_by_pct <- list()

for (spec in coverage_specs) {
  pct_rds_path <- file.path(output_dir, "CLAMPbase", sprintf("results_pct%d_msigdb_decoupler_gsea.rds", spec$pct))

  if (file.exists(pct_rds_path)) {
    message(sprintf("Loading cached pct-level result: CLAMPbase %d%%", spec$pct))
    results_clampbase_by_pct[[as.character(spec$pct)]] <- readRDS(pct_rds_path)
    next
  }

  seeds_to_run <- if (spec$pct == 100) 1 else seeds

  pct_rows <- lapply(seeds_to_run, function(s) {
    seed_dir   <- file.path(models_dir, spec$dir, sprintf("hall_coverage_rs%d_seed_%d", spec$pct, s))
    z_path     <- file.path(seed_dir, "CLAMPbase", "Z.csv")
    cache_path <- file.path(output_dir, "CLAMPbase", sprintf("rs%d_seed%d_msigdb_decoupler_gsea.rds", spec$pct, s))

    if (!file.exists(z_path)) { warning("Z.csv not found: ", z_path); return(NULL) }

    if (file.exists(cache_path)) {
      message(sprintf("Loading cached: CLAMPbase rs%d seed%d", spec$pct, s))
      res <- readRDS(cache_path)
    } else {
      message(sprintf("Running decoupleR GSEA: CLAMPbase rs%d seed%d", spec$pct, s))
      res <- run_gsea_for_model(z_path)
      if (!is.null(res)) saveRDS(res, cache_path)
    }
    if (is.null(res)) return(NULL)

    data.frame(
      model_type     = "CLAMPbase",
      coverage_pct   = spec$pct,
      seed           = s,
      n_samples      = res$n_samples,
      n_lvs          = res$n_lvs,
      n_total_msigdb = res$n_total_msigdb,
      stringsAsFactors = FALSE
    )
  })

  pct_df <- do.call(rbind, Filter(Negate(is.null), pct_rows))
  rownames(pct_df) <- NULL
  results_clampbase_by_pct[[as.character(spec$pct)]] <- pct_df
  saveRDS(pct_df, pct_rds_path)
  message(sprintf("Saved: CLAMPbase %d%% -> %s", spec$pct, pct_rds_path))
}

results_clampbase_df <- do.call(rbind, results_clampbase_by_pct)
rownames(results_clampbase_df) <- NULL
print(results_clampbase_df)

In [ ]:
for (pct in names(results_clampbase_by_pct)) {
  csv_path <- file.path(output_dir, "CLAMPbase", sprintf("results_pct%s_msigdb_decoupler_gsea.csv", pct))
  write.csv(results_clampbase_by_pct[[pct]], csv_path, row.names = FALSE)
  message("Saved: ", csv_path)
}